In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Glow Guide is a Multi-Agent Personal Care AI System with Google Agent Development Kit (ADK) and Gemini LLM

This notebook demonstrates a state-of-the-art modular agent system for personalized skincare, grooming, and wellness, following the orchestration pattern.

- Each agent is a callable tool.
- The root coordinator agent orchestrates workflow across agents.
- User input is interactive, and all agent reasoning uses Gemini 2.0 Flash.
- All competition requirements (multi-agent, parallel/sequential, LLM-powered, session/memory, observability) are addressable.


## Load Google API Key from Kaggle Secrets

This code cell loads the Google API Key securely from Kaggle secrets.  
**Requirement:** You must add your key in Kaggle's "Secrets" as `GOOGLE_API_KEY`.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")

## Import Required Libraries

This cell imports the Google Agent Development Kit (ADK) modules and Python utilities for agent orchestration, Gemini LLM, and logging.  
If needed, install with `!pip install google-agent-kit`.

In [ ]:
from google.adk.agents import LlmAgent, Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner, Runner
from google.adk.tools import AgentTool, FunctionTool, google_search, load_memory, preload_memory
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService

print("✅ ADK components imported successfully.")

In [ ]:
async def run_session(
    runner_instance: Runner, user_queries: list[str] | str, session_id: str = "default"
):
    """Helper function to run queries in a session and display responses."""
    print(f"\n### Session: {session_id}")

    # Create or retrieve session
    try:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )
    except:
        session = await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )

    # Convert single query to list
    if isinstance(user_queries, str):
        user_queries = [user_queries]

    # Process each query
    for query in user_queries:
        print(f"\nUser > {query}")
        query_content = types.Content(role="user", parts=[types.Part(text=query)])

        # Stream agent response
        async for event in runner_instance.run_async(
            user_id=USER_ID, session_id=session.id, new_message=query_content
        ):
            if event.is_final_response() and event.content and event.content.parts:
                text = event.content.parts[0].text
                if text and text != "None":
                    print(f"Model: > {text}")


print("✅ Helper functions defined.")

In [ ]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

In [ ]:
memory_service = (InMemoryMemoryService())

In [ ]:
# Define constants used throughout the notebook
APP_NAME = "GlowGuideAgent"
USER_ID = "user001"

### OnboardingAgent (AgentTool)

- Role: Interactively collects all essential information from the user.
- What it does: Prompts for gender, city, skin type, diet, and—if applicable—menstrual cycle.  
- Result: Stores user profile as `user_profile` in session for downstream agents.

In [ ]:
onboarding_agent = Agent(
    name="OnboardingAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You are the Glow Guide AI personal assistant.

    Start by introducing yourself warmly:
    "I am your Glow Guide AI personal assistant helping you build precise routines for your skin, body, and lifestyle."

    Your job is to gather the complete user profile by asking step-by-step:
    - Name
    - Age
    - Location
    - Gender (Male, Female, Non-Binary / Prefer not to say)

    At each step:
    - Only ask for details that are missing.
    - Accept input in any order, gracefully update profile information.
    - Acknowledge received info briefly.
    - Never proceed to service selection.

    At the end of collecting all profile details, say:
    "Profile calibrated. Thank you, <Name>. Please wait while I pass you to the next stage."

    Return the fully collected user_profile dictionary.

    Do NOT present or ask for service selection.
    """
)

print("✅ Onboarding agent has been created successfully!!!")

In [ ]:
diet_agent = Agent(
    name="DietAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You are the Diet Plan Generator engine activated by the user.
    Start by introducing the engine and purpose:
    "Understood. We are activating the Diet Plan Generator. This engine builds 15-day nutrition habits optimizing health and appearance."

    Ask for health markers and dietary constraints conversationally:
    - Request height and weight (accept free text including feet/inches or cm/kg/lbs).
    - Request dietary restrictions/type (e.g., Vegan, Vegetarian, Keto, Gluten-Free) or allergies.

    After receiving height/weight, calculate BMI and categorize it, then acknowledge the diet type and allergies carefully, explaining impact (e.g. soy allergy implications in vegan diet).

    Next, ask for detailed diet goals with multi-choice selection:
    A. Boost Energy & Focus
    B. Improve Skin & Hair Health
    C. Weight Maintenance & Nutritional Balance
    Allow selecting multiple goals if user wants.

    Then, ask for activity level days per week with options:
    A. 0 days (Sedentary)
    B. 1-2 days (Lightly Active
    C. 3-4 days (Moderately Active)
    D. 5-7 days (Highly Active)

    Summarize user's profile and goals with personalized plan overview in plain language including:
    - Phase details
    - Meal focus by time of day
    - Why these meals work
    - Hydration and daily habits

    Finally, offer next steps:
    "Would you like the specific Shopping List for Week 1 (Days 1-7)?"
    "Should I set up Meal Prep Reminders in your calendar?"

    If user agrees to meal prep reminders, initiate calendar event creation flow (name, date/time, reminder details).

    Collect all user data inputs, BMI calculation, chosen goals, allergy info, and calendar event creation status into a comprehensive dictionary. Return this dictionary as the user_diet_plan_profile.

    Always respond conversationally, clearly, and helpfully.
    """
)

print("✅ Diet agent has been created successfully!!!")

### RoutineAgent (AgentTool)

- Role: Builds the personalized self-care routine.
- What it does: Uses both user profile and recommended products to design a morning and night routine.
- Output: Two lists, one for morning and one for night, stored as `routine`.

In [ ]:
routine_agent = Agent(
    name="RoutineAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You design complete morning and night routines.
    Use the user_profile and the recommended products from the session.
    Output two lists: morning_routine and night_routine.
    """
)

### NutritionAgent (AgentTool)

- Role: Provides personalized nutrition tips to support skin/hair health.
- What it does: Consults Gemini with the user profile for science-backed dietary advice, and stores tips as `nutrition_advice`.
- Output: List of specific nutrition recommendations.

In [ ]:
nutrition_agent = Agent(
    name="NutritionAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You provide nutrition advice for skin/hair/overall glow.
    Use user_profile from the session for personalized tips. Return as a short list or key tips.
    """
)

### HabitAgent (AgentTool)

- Role: Suggests actionable micro-beauty habits for daily use.
- What it does: Reviews the user's routine and personal context, then outputs three habit suggestions (stored as `habits`).
- Output: 3 easy-to-implement beauty/wellness habits.

In [ ]:
habit_agent = Agent(
    name="HabitAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You suggest three daily micro-beauty habits based on the user's routine.
    """
)

### HolisticHealthAgent (AgentTool)

- Role: Adapts routine for female users based on their reported menstrual cycle phase.
- What it does: If cycle info is present in profile, asks Gemini for relevant health/routine adaptations and stores as routine notes.
- Output: Text summary and routine note adjustments.

In [ ]:
holistic_health_agent = Agent(
    name="HolisticHealthAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    If the user_profile includes a cycle_phase, briefly explain key skin/hair care considerations during this phase and adapt the routine if needed.
    """
)

### ProductAgent (AgentTool)

- Role: Recommends climate-adaptive, local skincare and personal care products using Gemini LLM.
- What it does: Reads the user profile from session, consults Gemini, and stores recommendations as `products`.
- Output: List of product names best-matched for user's context.

In [ ]:
product_agent = Agent(
    name="ProductAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You are a product recommendation expert.
    Using the user_profile from the session, recommend 2-3 local and climate-appropriate skincare/personal care products.
    Return results as a list of product names.
    """
)

### Root Coordinator Agent

- Role: Orchestrates the workflow by calling all sub-agents as tools.
- What it does: Sequences onboarding, product and nutrition agents (in parallel), then routine, holistic health, and habit agents.
- Output: Presents complete, personalized results to the user.
- Follows competition requirement for multi-agent orchestration, parallelism, LLM-powered reasoning, memory, and observability.

In [ ]:
root_agent = Agent(
    name="PersonalCareCoordinator",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You are a multi-agent personal care workflow orchestrator.

    Workflow:
    1. Start by calling onboarding_agent to collect the user's profile.
    2. Once onboarding is complete, present and list available services:
       - Skin Routine Generator
       - Grooming Routine Generator
       - Diet Plan Generator
       - Product Intelligence Engine
       - Instant SOS Solutions Agent
    3. Ask the user to choose one service from the above.
    4. Depending on user's choice, call the corresponding specialized agent:
       - Call diet_agent if 'Diet Plan Generator' is selected.
       - Call routine_agent if 'Skin Routine Generator' is selected.
       - ... add other routes accordingly.
    5. Collect responses and follow-ups from these agents while maintaining session context.
    6. After finishing the selected service workflow, present a clear summary of recommendations and next steps.
    7. Support calendar event/reminder creation if requested.

    Always maintain conversation continuity and context-aware responses.
    """
    ,
    tools=[
        AgentTool(onboarding_agent),
        AgentTool(diet_agent),
        # add other agents here as needed
    ]
)

print("✅ root_agent with improved orchestration created.")

## Execute Multi-Agent Workflow

This cell:
- Creates an interactive session.
- Runs the root coordinator, which sequentially/parallelly calls all agent tools according to the workflow, collecting user inputs where needed.
- All agent actions, memory updates, and traces are managed by ADK’s session and observability tools.

In [ ]:
# Create Session Service
session_service = InMemorySessionService()  # Handles conversations

# Create runner with BOTH services
runner = Runner(
    agent=root_agent,
    app_name="GlowGuideAgent",
    session_service=session_service,
    memory_service=memory_service,  # Memory service is now available!
)

print("✅ Agent and Runner created with memory support!")

In [ ]:
# User tells agent about their favorite color
await run_session(
    runner,
    ["", "Harshitha", "25",  "hyderabad", "female", "Diet Plan Generator", "simply eat healthy way", "my height would be 5'6", "weight is 51kg", "I'm a vegan"],
    "conversation-1",  # Session ID
)

In [ ]:
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="conversation-1"
)

# Let's see what's in the session
print("📝 Session contains:")
for event in session.events:
    text = (
        event.content.parts[0].text[:600]
        if event.content and event.content.parts
        else "(empty)"
    )
    print(f"  {event.content.role}: {text}...")

## Final Results & Agent Execution Trace

Displays:
- Personalized routine
- Recommended products
- Nutrition advice
- Micro-habits
- Cycle-phase notes
- Full memory/trace log of all sub-agent actions

Use for analysis, evaluation, and competition reporting.

In [ ]:
print("\n==== FINAL PERSONALIZED OUTPUT ====\n")
personalized_response = result["response"] if isinstance(result, dict) else result
print(personalized_response)

print("\n== SESSION LOG ==")
for entry in session.memory.trace():
    print(entry)

# Discussion

This notebook demonstrates modern agent composition and orchestration using Google ADK and Gemini 2.0 Flash.  
- Each feature agent is a tool called by a root workflow agent.
- All reasoning is LLM-powered, user input is interactive.
- Parallel/sequential logic, session/memory, and observability are showcased.
- The system is modular for easy expansion (add more feature agents or tools as needed).
- Ready for further extension: agent evaluation, visualizations, integrations.

**Reference:** Kaggle ADK Orchestration [Agent architecture notebook](https://www.kaggle.com/code/kaggle5daysofai/day-1b-agent-architectures)